In [ ]:
import json
from datetime import datetime
from matplotlib import pyplot as plt
import pycountry

from emu_renewal.inputs import DATA_PATH, get_hosp_series_from_owid_data, get_country_hosps, \
    find_null_data, find_neg_data, has_change_repeated, has_outlier

In [ ]:
included = json.load(open(DATA_PATH / "config/included.json", "r"))
start_time = datetime(2020, 4, 1)
end_time = datetime(2021, 6, 30)

In [ ]:
hosp_data = {}
ind = {}
for c in included:
    data, indicator = get_country_hosps(c, start_time, end_time)
    hosp_data[c] = data
    ind[c] = indicator

In [ ]:
# Demonstrate that the hospitalisation data meet the same criteria as we applied to cases and deaths
hosp_present = {c: d for c, d in hosp_data.items() if d is not None}
print(len(find_null_data(hosp_present)))
print(len(find_neg_data(hosp_present)))
print(any([has_change_repeated(d, 6) for d in hosp_present.values()]))
print(any([has_outlier(d) for d in hosp_present.values()]))

In [ ]:
fig, axes = plt.subplots(10, 9, figsize=(20, 20))
flat_axes = axes.ravel()
for i, (c, data) in enumerate(hosp_data.items()):
    ax = flat_axes[i]
    title = pycountry.countries.lookup(c).name
    if data is not None:
        ax.plot(data.index, data)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
        title = title + f", {ind[c]}"
    ax.set_title(title)
fig.tight_layout()